In [1]:
import os

# Mobaxterm으로 linux 서버 직접 접속해서 DIPLAY 환경변수 확인하고 그 내용을 직접 할당
os.environ["DISPLAY"]="localhost:13.0"
print(os.environ.get("DISPLAY"))

from keysight.ads import de
from keysight.ads.de import db_uu as db
from keysight.edatoolbox import circuit
from keysight.edatoolbox import ads
import keysight.ads.dataset as dataset

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

import numpy as np

from stable_baseline_PPO import AdsCircuitEnv, CircuitSpec, AdsRunnerConfig, SparamCSVLogger

param_init = {"general_TL" : np.array([200e-6, 200e-6, 200e-6, 200e-6, 200e-6, 200e-6, 200e-6, 200e-6, 200e-6, 200e-6, 200e-6], dtype=np.float64),
                "bias_TL" : np.array([1000e-6, 1000e-6, 1000e-6, 1000e-6], dtype=np.float64),
                "ELC" : np.array([50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6, 50e-6], dtype=np.float64)
                }

param_low = {"general_TL" : np.array([9e-6, 9e-6, 9e-6, 9e-6, 9e-6, 9e-6, 9e-6, 9e-6, 9e-6, 9e-6, 9e-6], dtype=np.float64),
                "bias_TL" : np.array([700e-6, 700e-6, 700e-6, 700e-6], dtype=np.float64),
                "ELC" : np.array([16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6, 16e-6], dtype=np.float64)
                }
param_high = {"general_TL" : np.array([500e-6, 500e-6, 500e-6, 500e-6, 500e-6, 500e-6, 500e-6, 500e-6, 500e-6, 500e-6, 500e-6], dtype=np.float64),
                "bias_TL" : np.array([1800e-6, 1800e-6, 1800e-6, 1800e-6], dtype=np.float64),
                "ELC" : np.array([100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6, 100e-6], dtype=np.float64)
                }

csv_logger = SparamCSVLogger(
    out_dir="/home/jychung/python/target_spec/reward_max_calc"
)

cell_name = "sample_gan_25GHz_twostage_ppo"

spec = CircuitSpec(
    s11_db_target=-10.0,
    s22_db_target=-10.0,
    NF_db_target=2.0,
    f_min_ghz=12.0e9,
    f_max_ghz=18.0e9,
)

workspace_path = "/home/jychung/ADS_project/test"
workspace = de.open_workspace(workspace_path)

library_name = "tutorial1_lib"
design = db.open_design(f"tutorial1_lib:{cell_name}:schematic", db.DesignMode.APPEND)


runner = AdsRunnerConfig(
    ADS_sim_output_dir="/home/jychung/ADS_project/test/sim_logfile/"
)

# 기존과 동일하게 design/spec/runner/params 준비하고 make_env 재사용
vec_env = DummyVecEnv([lambda: Monitor(AdsCircuitEnv(
    design=design, spec=spec, runner=runner, csv_logger=csv_logger,
    param_init=param_init, param_low=param_low, param_high=param_high,
    max_steps=25, action_scale=0.08, seed=1
))])

model = PPO.load("/home/jychung/python/model_saved/ppo_ads_circuit2.zip", env=vec_env, device="cpu")

obs = vec_env.reset()
for _ in range(25):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    if done[0]:
        print("meets:", info[0].get("meets"), "reward:", reward[0])
        break


localhost:13.0


Could not create collator: 4


localhost:13.0


RuntimeError: Unable to lock database file for tutorial1_lib/sample_gan_25GHz_twostage_ppo/schematic.